In [29]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [30]:
load_dotenv()

True

In [31]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [32]:
model.invoke("Distance of moon from earth")

AIMessage(content='The distance of the Moon from Earth is not constant because the Moon orbits Earth in an **elliptical path**, not a perfect circle.\n\nHere are the key distances:\n\n*   **Average Distance:** Approximately **384,400 kilometers (238,900 miles)**\n*   **Perigee (Closest Point):** Approximately **363,104 kilometers (225,623 miles)**\n*   **Apogee (Farthest Point):** Approximately **405,696 kilometers (252,088 miles)**\n\nThis variation means that the Moon can appear slightly larger or smaller in our sky depending on where it is in its orbit.\n\nAdditionally, due to tidal forces, the Moon is very slowly drifting away from Earth at a rate of about 3.8 centimeters (1.5 inches) per year.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d8cc3-5cf9-71c0-9e0b-6663d31939f9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output

In [33]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str
    seo_report:str
    clarity_report:str
    engagement_report:str
    summary:str

In [34]:
graph = StateGraph(BlogState)

In [35]:
def create_outline(state:BlogState)->BlogState:
    topic=state['title']
    prompt=f"Create detailed outline for a blog on topic: {topic}"
    outline = model.invoke(prompt).content
    state['outline']=outline
    return state

def create_content(state:BlogState)->BlogState:
    title=state['title']
    outline=state['outline']
    prompt=f"Create a detailed blog post on {title} using the following outline: {outline}"
    content=model.invoke(prompt).content
    state['content']=content
    return state

In [36]:
def check_seo(state:BlogState):
  topic = state['title']
  content = state['content']
    
  prompt = f"You are a professional blog content analyzer. Analyze this blog on the topic: {topic} for SEO: {content}. Check for headers and keyword flow."
  response = model.invoke(prompt) 
  return {"seo_report": response}

def clarity_critic(state: BlogState):
  """Checks for fluff, repetition, and logical clarity."""
  topic = state['title']
  outline = state['outline']
  content = state['content']
  prompt = (
    f"You are a professional blog content copyeditor. Review this blog on topic {topic}: {content}\n"
    "Identify:\n"
    "1. Redundancy: Are there sentences that say the same thing twice?\n"
    "2. Jargon: Are there overly complex words that could be simpler?\n"
    "3. Flow: Does each point lead naturally to the next?\n"
    "Provide a 'Clarity Score' (1-10) and suggestions for cutting the fluff."
  )
  
  response = model.invoke(prompt)
  return {"clarity_report": response.content}

def engagement_critic(state: BlogState):
  """Assesses if the post is boring or actually provides value to a human."""

  topic = state['title']
  outline = state['outline']
  content = state['content']
  prompt = (
    f"You are a critical blog content reader. Analyze this blog post on the topic {topic}: content\n"
    "Rate 1-10 on: \n"
    "1. Hook: Does the first paragraph grab attention?\n"
    "2. Actionability: Can the reader actually do something with this info?\n"
    "3. CTA: Is the ending persuasive?\n"
    "Return a brief summary of what could be better."
  )
  # Simple LLM call
  response = model.invoke(prompt)
  return {"engagement_report": response.content}

In [37]:
def summary(state:BlogState):
    """Aggregate and show the blog content and analysis"""
    # topic = state['title']
    # # outline = state['outline']
    # # content = state['content']
    seo_report = state['seo_report']
    clarity_report = state['clarity_report']
    engagement_report = state['engagement_report']
    summary = f"""
    Please find the analysis:\n
    SEO Report: {seo_report}
    Clarity_report: {clarity_report}
    Engagement Report: {engagement_report}
    """
    state['summary'] = summary
    return state

In [ ]:
graph.add_node('create_outline', create_outline)
graph.add_node('create_content', create_content)
graph.add_node('check_seo', check_seo)
graph.add_node('clarity_critic', clarity_critic)
graph.add_node('engagement_critic', engagement_critic)
graph.add_node('summary',summary)


In [39]:
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_content')
graph.add_edge('create_content', 'check_seo')
graph.add_edge('create_content', 'clarity_critic')
graph.add_edge('create_content', 'engagement_critic')
graph.add_edge("check_seo", "summary")
graph.add_edge("clarity_critic", "summary")
graph.add_edge("engagement_critic", "summary")
graph.add_edge('summary', END)


In [40]:
workflow = graph.compile()

ValueError: Found edge starting at unknown node 'summary'

In [13]:
workflow

NameError: name 'workflow' is not defined

In [17]:
initial_state={'title':'AI in India'}
final_state=workflow.invoke(initial_state)
print(final_state)

{'title': 'AI in India', 'outline': 'Here\'s a detailed outline for a blog post on "AI in India," designed to be comprehensive, engaging, and informative for a broad audience.\n\n---\n\n## Blog Title Options:\n\n*   **India\'s AI Ascent: Unpacking the Revolution Shaping a Nation** (Chosen for outline)\n*   The Digital Awakening: How AI is Redefining India\'s Future\n*   Beyond the Hype: A Deep Dive into AI\'s Impact in India\n*   From Startups to Scale: Charting India\'s AI Journey\n*   India\'s Intelligence Boom: Why the World is Watching Its AI Evolution\n\n---\n\n## Detailed Blog Outline: India\'s AI Ascent: Unpacking the Revolution Shaping a Nation\n\n### **I. Introduction (Approx. 150-200 words)**\n\n*   **A. Catchy Hook:** Start with a compelling statistic or a vivid image of India\'s digital transformation and its scale. (e.g., "From bustling mega-cities to remote villages, India is undergoing a profound digital metamorphosis...")\n*   **B. Introduce AI\'s Role:** Briefly define

In [19]:
print(final_state['title'])

AI in India


In [20]:
print(final_state['outline'])

Here's a detailed outline for a blog post on "AI in India," designed to be comprehensive, engaging, and informative for a broad audience.

---

## Blog Title Options:

*   **India's AI Ascent: Unpacking the Revolution Shaping a Nation** (Chosen for outline)
*   The Digital Awakening: How AI is Redefining India's Future
*   Beyond the Hype: A Deep Dive into AI's Impact in India
*   From Startups to Scale: Charting India's AI Journey
*   India's Intelligence Boom: Why the World is Watching Its AI Evolution

---

## Detailed Blog Outline: India's AI Ascent: Unpacking the Revolution Shaping a Nation

### **I. Introduction (Approx. 150-200 words)**

*   **A. Catchy Hook:** Start with a compelling statistic or a vivid image of India's digital transformation and its scale. (e.g., "From bustling mega-cities to remote villages, India is undergoing a profound digital metamorphosis...")
*   **B. Introduce AI's Role:** Briefly define AI and immediately connect it to India's unique context – its va

In [21]:
print(final_state['content'])

## India's AI Ascent: Unpacking the Revolution Shaping a Nation

From bustling mega-cities teeming with digital natives to remote villages embracing smartphone connectivity, India is undergoing a profound digital metamorphosis. This transformation isn't just about adopting technology; it's about fundamentally reshaping how a nation of 1.4 billion people lives, works, and interacts. At the heart of this revolution lies Artificial Intelligence (AI) – a powerful suite of technologies enabling machines to learn, reason, and solve problems like humans. In India's unique context, AI isn't merely an upgrade; it's a strategic imperative, offering unprecedented opportunities to tackle complex challenges, drive inclusive growth, and unlock immense potential.

This blog post will delve into why India is rapidly emerging as a significant global player in AI, driven by unique demographic, technological, and governmental factors. We'll explore the current landscape, highlight key sectors being trans